# 13. Managing States (save & load a conversation)

A **state** is a snapshot of everything an agent or team **remembers right now** —
the whole conversation so far.

We can **save** the state to a file and **load** it back later, so the agent can
**continue where it left off**, even after the program closes.

## Real-life analogy

Saving state is like **saving a video game**.

- **Save** = write your progress to a file.
- **Close the game** (the program ends).
- **Load** = open the file later and continue from the exact same point.

## The operations

| Operation | Code |
|-----------|------|
| Save | `state = await agent.save_state()` |
| Write to file | `json.dump(state, open("state.json", "w"))` |
| Read from file | `state = json.load(open("state.json"))` |
| Load | `await agent.load_state(state)` |

(Teams have the **same** `save_state` / `load_state` methods.)

In [1]:
from dotenv import load_dotenv
load_dotenv()

import json
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
agent = AssistantAgent(name="assistant", model_client=model_client,
                       system_message="You are a helpful assistant with a good memory.")

# Tell the agent a fact in this conversation
await agent.run(task="Remember that my favourite number is 7.")

# SAVE the state to a file
state = await agent.save_state()
with open("agent_state.json", "w") as f:
    json.dump(state, f)
print("Saved the conversation to agent_state.json")

Saved the conversation to agent_state.json


In [2]:
# Imagine the program closed. Now we make a NEW agent and LOAD the saved state.
new_agent = AssistantAgent(name="assistant", model_client=model_client,
                           system_message="You are a helpful assistant with a good memory.")

with open("agent_state.json") as f:
    saved = json.load(f)

await new_agent.load_state(saved)     # load the old conversation

# It should still remember the favourite number
result = await new_agent.run(task="What is my favourite number?")
print(result.messages[-1].content)

Your favorite number is 7.


## Key points to remember

- **State** = a snapshot of everything the agent/team remembers (the conversation).
- **Save** with `await agent.save_state()`; **load** with `await agent.load_state(state)`.
- Write the state to a **file** (JSON) to keep it after the program closes.
- Load it later so the agent **continues where it left off** (like a game save).
- **Teams** use the same `save_state` / `load_state` methods.